In [80]:

pip install pyspark


Note: you may need to restart the kernel to use updated packages.


Stablishing a SparkSession

In [46]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, max, count, desc, row_number
from pyspark.sql.window import Window
import os

spark = SparkSession.builder \
    .appName("CampusAnalytics") \
    .master("local[*]") \
    .getOrCreate()

print("Spark started successfully!")
print("Spark version:", spark.version)


Spark started successfully!
Spark version: 4.2.0


Reading the data from the files

In [79]:

# Files should be in the same folder as your notebook

students_df = spark.read.csv(
    "students.csv",
    header=True,
    inferSchema=True
)

attendance_df = spark.read.csv(
    "attendance.csv",
    header=True,
    inferSchema=True
)

budget_df = spark.read.option(
    "multiline", "true"
).json("department_budget.json")

print("All three files loaded successfully!")


All three files loaded successfully!


In [1]:
import os
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot


In [2]:
import os

os.environ["JAVA_HOME"] = (
    r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
)

ADDING ALL THE THREE FILES

In [6]:

# Files should be in the same folder as your notebook

students_df = spark.read.csv(
    "students.csv",
    header=True,
    inferSchema=True
)

attendance_df = spark.read.csv(
    "attendance.csv",
    header=True,
    inferSchema=True
)

budget_df = spark.read.option(
    "multiline", "true"
).json("department_budget.json")

print("All three files loaded successfully!")


All three files loaded successfully!


INSPECT SCHEMA AND DATAS

In [7]:

print("STUDENTS")
students_df.printSchema()
students_df.show(5, truncate=False)

print("ATTENDANCE")
attendance_df.printSchema()
attendance_df.show(5, truncate=False)

print("DEPARTMENT BUDGET")
budget_df.printSchema()
budget_df.show(5, truncate=False)


STUDENTS
root
 |-- student_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- city: string (nullable = true)
 |-- marks: integer (nullable = true)

+----------+------+----------+-------+-----+
|student_id|name  |department|city   |marks|
+----------+------+----------+-------+-----+
|201       |Aarav |CSE       |Delhi  |84   |
|202       |Diya  |IT        |Lucknow|76   |
|203       |Kabir |CSE       |Kanpur |91   |
|204       |Anaya |ECE       |Delhi  |68   |
|205       |Ishaan|IT        |Noida  |79   |
+----------+------+----------+-------+-----+
only showing top 5 rows
ATTENDANCE
root
 |-- student_id: integer (nullable = true)
 |-- attendance: integer (nullable = true)

+----------+----------+
|student_id|attendance|
+----------+----------+
|201       |92        |
|202       |85        |
|203       |96        |
|204       |78        |
|205       |88        |
+----------+----------+
only showing top 5 rows
DEPARTMENT BUDGET


JOINING ALL THE DATASETS 

In [47]:

# Join students and attendance using student_id
combined_df = students_df.join(
    attendance_df,
    on="student_id",
    how="left"
)

# Join departmental budgets using department
combined_df = combined_df.join(
    budget_df,
    on="department",
    how="left"
)

combined_df.show(truncate=False)
print("Joined dataset created successfully!")


+----------+----------+-------+-------+-----+----------+-----------------+
|department|student_id|name   |city   |marks|attendance|department_budget|
+----------+----------+-------+-------+-----+----------+-----------------+
|CSE       |201       |Aarav  |Delhi  |84   |92        |500000           |
|IT        |202       |Diya   |Lucknow|76   |85        |350000           |
|CSE       |203       |Kabir  |Kanpur |91   |96        |500000           |
|ECE       |204       |Anaya  |Delhi  |68   |78        |275000           |
|IT        |205       |Ishaan |Noida  |79   |88        |350000           |
|CSE       |206       |Sara   |Lucknow|88   |93        |500000           |
|ECE       |207       |Vivaan |Kanpur |72   |81        |275000           |
|IT        |208       |Myra   |Delhi  |83   |90        |350000           |
|CSE       |209       |Advik  |Noida  |95   |98        |500000           |
|ECE       |210       |Kiara  |Lucknow|74   |86        |275000           |
|CSE       |211       |Re

Create performance categories and filter students

In [12]:

analytics_df = combined_df.withColumn(
    "performance_category",
    when(col("marks") >= 90, "Outstanding")
    .when(col("marks") >= 80, "Excellent")
    .when(col("marks") >= 70, "Good")
    .otherwise("Needs Improvement")
)

analytics_df.select(
    "student_id",
    "name",
    "department",
    "marks",
    "performance_category"
).show(truncate=False)

# Filter students scoring at least 85
high_scorers_df = analytics_df.filter(
    col("marks") >= 85
)

print("Students scoring at least 85:")
high_scorers_df.show(truncate=False)


+----------+-------+----------+-----+--------------------+
|student_id|name   |department|marks|performance_category|
+----------+-------+----------+-----+--------------------+
|201       |Aarav  |CSE       |84   |Excellent           |
|202       |Diya   |IT        |76   |Good                |
|203       |Kabir  |CSE       |91   |Outstanding         |
|204       |Anaya  |ECE       |68   |Needs Improvement   |
|205       |Ishaan |IT        |79   |Good                |
|206       |Sara   |CSE       |88   |Excellent           |
|207       |Vivaan |ECE       |72   |Good                |
|208       |Myra   |IT        |83   |Excellent           |
|209       |Advik  |CSE       |95   |Outstanding         |
|210       |Kiara  |ECE       |74   |Good                |
|211       |Reyansh|CSE       |81   |Excellent           |
|212       |Aadhya |IT        |89   |Excellent           |
|213       |Arjun  |ECE       |65   |Needs Improvement   |
|214       |Ira    |CSE       |93   |Outstanding        

In [18]:
analytics_df.show()

+----------+----------+-------+-------+-----+----------+-----------------+--------------------+
|department|student_id|   name|   city|marks|attendance|department_budget|performance_category|
+----------+----------+-------+-------+-----+----------+-----------------+--------------------+
|       CSE|       201|  Aarav|  Delhi|   84|        92|           500000|           Excellent|
|        IT|       202|   Diya|Lucknow|   76|        85|           350000|                Good|
|       CSE|       203|  Kabir| Kanpur|   91|        96|           500000|         Outstanding|
|       ECE|       204|  Anaya|  Delhi|   68|        78|           275000|   Needs Improvement|
|        IT|       205| Ishaan|  Noida|   79|        88|           350000|                Good|
|       CSE|       206|   Sara|Lucknow|   88|        93|           500000|           Excellent|
|       ECE|       207| Vivaan| Kanpur|   72|        81|           275000|                Good|
|        IT|       208|   Myra|  Delhi| 

Department-wise aggregations

In [25]:
from pyspark.sql.functions import avg, max, count, round

department_summary = analytics_df.groupBy("department").agg(
    round(avg("marks"), 2).alias("average_marks"),
    max("marks").alias("maximum_marks"),
    count("student_id").alias("student_count"),
    round(avg("attendance"), 2).alias("percentage_attendance")
)

department_summary.show()

+----------+-------------+-------------+-------------+---------------------+
|department|average_marks|maximum_marks|student_count|percentage_attendance|
+----------+-------------+-------------+-------------+---------------------+
|       ECE|        74.17|           86|            6|                82.83|
|        IT|         82.0|           94|            6|                 89.5|
|       CSE|         87.0|           95|            8|                92.88|
+----------+-------------+-------------+-------------+---------------------+



Spark SQL queries

In [26]:

# Create the required temporary view
analytics_df.createOrReplaceTempView("student_analytics")

# Query 1: Students scoring at least 85
spark.sql("""
    SELECT *
    FROM student_analytics
    WHERE marks >= 85
    ORDER BY marks DESC
""").show(truncate=False)

# Query 2: Average marks by department
spark.sql("""
    SELECT department, AVG(marks) AS average_marks
    FROM student_analytics
    GROUP BY department
    ORDER BY average_marks DESC
""").show(truncate=False)

# Query 3: Three highest-scoring students
spark.sql("""
    SELECT student_id, name, department, marks
    FROM student_analytics
    ORDER BY marks DESC
    LIMIT 3
""").show(truncate=False)


+----------+----------+------+-------+-----+----------+-----------------+--------------------+
|department|student_id|name  |city   |marks|attendance|department_budget|performance_category|
+----------+----------+------+-------+-----+----------+-----------------+--------------------+
|CSE       |209       |Advik |Noida  |95   |98        |500000           |Outstanding         |
|IT        |218       |Navya |Lucknow|94   |96        |350000           |Outstanding         |
|CSE       |214       |Ira   |Lucknow|93   |97        |500000           |Outstanding         |
|CSE       |203       |Kabir |Kanpur |91   |96        |500000           |Outstanding         |
|IT        |212       |Aadhya|Delhi  |89   |95        |350000           |Excellent           |
|CSE       |206       |Sara  |Lucknow|88   |93        |500000           |Excellent           |
|CSE       |220       |Pari  |Delhi  |87   |94        |500000           |Excellent           |
|ECE       |216       |Anvi  |Delhi  |86   |91    

Rank students using a window function

In [27]:

department_window = Window.partitionBy(
    "department"
).orderBy(
    col("marks").desc(),
    col("student_id").asc()
)

ranked_df = analytics_df.withColumn(
    "department_rank",
    row_number().over(department_window)
)

print("All students with department ranks:")
ranked_df.select(
    "student_id",
    "name",
    "department",
    "marks",
    "department_rank"
).show(truncate=False)

# Top two students from each department
top_two_df = ranked_df.filter(
    col("department_rank") <= 2
)

print("Top two students per department:")
top_two_df.select(
    "student_id",
    "name",
    "department",
    "marks",
    "department_rank"
).show(truncate=False)


All students with department ranks:
+----------+-------+----------+-----+---------------+
|student_id|name   |department|marks|department_rank|
+----------+-------+----------+-----+---------------+
|209       |Advik  |CSE       |95   |1              |
|214       |Ira    |CSE       |93   |2              |
|203       |Kabir  |CSE       |91   |3              |
|206       |Sara   |CSE       |88   |4              |
|220       |Pari   |CSE       |87   |5              |
|201       |Aarav  |CSE       |84   |6              |
|211       |Reyansh|CSE       |81   |7              |
|217       |Krish  |CSE       |77   |8              |
|216       |Anvi   |ECE       |86   |1              |
|219       |Dhruv  |ECE       |80   |2              |
|210       |Kiara  |ECE       |74   |3              |
|207       |Vivaan |ECE       |72   |4              |
|204       |Anaya  |ECE       |68   |5              |
|213       |Arjun  |ECE       |65   |6              |
|218       |Navya  |IT        |94   |1        

Final Verification

In [53]:

print("Total records:", analytics_df.count())

print(
    "Missing student IDs:",
    analytics_df.filter(col("student_id").isNull()).count()
)

print(
    "Missing marks:",
    analytics_df.filter(col("marks").isNull()).count()
)

print(
    "Number of departments:",
    analytics_df.select("department").distinct().count()
)

print("Campus Analytics pipeline completed!")


Total records: 20
Missing student IDs: 0
Missing marks: 0
Number of departments: 3
Campus Analytics pipeline completed!
